Caso de estudio 2 - Benchmark de Formatos: Airline Delay & Cancellation Data

In [2]:

import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-avro:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.nio.file.{Files, Paths}
import java.io.File

val spark = SparkSession.builder()
  .appName("AeroMetrics_Benchmark")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .config("spark.sql.shuffle.partitions", "4")
  .getOrCreate()

import spark.implicits._

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo — AeroMetrics Analytics")

✅ Spark 3.5.0 listo — AeroMetrics Analytics


import $ivy.$
import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.nio.file.{Files, Paths}
import java.io.File
spark: SparkSession = org.apache.spark.sql.SparkSession@6ca06d11
import spark.implicits._

In [ ]:

def medirTiempo[T](bloque: => T): (T, Long) = {
  val inicio = System.nanoTime()
  val resultado = bloque
  val fin = System.nanoTime()
  val tiempoMs = (fin - inicio) / 1_000_000L
  (resultado, tiempoMs)
}


def tamanoBytes(ruta: String): Long = {
  val p = Paths.get(ruta)
  if (!Files.exists(p)) 0L
  else {
    Files.walk(p)
      .filter(Files.isRegularFile(_))
      .mapToLong(Files.size(_))
      .sum()
  }
}

// Formatea bytes a una unidad legible
def formatoTamano(bytes: Long): String = {
  if      (bytes < 1_024L)             f"$bytes B"
  else if (bytes < 1_048_576L)         f"${bytes / 1_024.0}%.1f KB"
  else if (bytes < 1_073_741_824L)     f"${bytes / 1_048_576.0}%.2f MB"
  else                                  f"${bytes / 1_073_741_824.0}%.2f GB"
}

println("✅ Funciones de benchmark definidas")

✅ Funciones de benchmark definidas


defined function medirTiempo
defined function tamanoBytes
defined function formatoTamano

In [ ]:

val rutaBase    = "C:/Curso-Scala/datos/airline"
val rutaCSVRaw  = s"$rutaBase/csv_raw"
val rutaSalida  = s"$rutaBase/salida"

val carpetas = List(
  s"$rutaSalida/consolidado_csv",
  s"$rutaSalida/parquet_sin_partir",
  s"$rutaSalida/orc",
  s"$rutaSalida/avro",
  s"$rutaSalida/parquet_particionado"
)

carpetas.foreach { c =>
  Files.createDirectories(Paths.get(c))
  println(s"  ✅ $c")
}

println(s"\n📁 Estructura de salida lista en: $rutaSalida")

  ✅ C:/Curso-Scala/datos/airline/salida/consolidado_csv
  ✅ C:/Curso-Scala/datos/airline/salida/parquet_sin_partir
  ✅ C:/Curso-Scala/datos/airline/salida/orc
  ✅ C:/Curso-Scala/datos/airline/salida/avro
  ✅ C:/Curso-Scala/datos/airline/salida/parquet_particionado

📁 Estructura de salida lista en: C:/Curso-Scala/datos/airline/salida


rutaBase: String = "C:/Curso-Scala/datos/airline"
rutaCSVRaw: String = "C:/Curso-Scala/datos/airline/csv_raw"
rutaSalida: String = "C:/Curso-Scala/datos/airline/salida"
carpetas: List[String] = List(
  "C:/Curso-Scala/datos/airline/salida/consolidado_csv",
  "C:/Curso-Scala/datos/airline/salida/parquet_sin_partir",
  "C:/Curso-Scala/datos/airline/salida/orc",
  "C:/Curso-Scala/datos/airline/salida/avro",
  "C:/Curso-Scala/datos/airline/salida/parquet_particionado"
)

In [ ]:

val schemaVuelos = StructType(List(
  StructField("FL_DATE",             StringType,  nullable = true),
  StructField("OP_CARRIER",          StringType,  nullable = true),
  StructField("OP_CARRIER_FL_NUM",   IntegerType, nullable = true),
  StructField("ORIGIN",              StringType,  nullable = true),
  StructField("DEST",                StringType,  nullable = true),
  StructField("CRS_DEP_TIME",        IntegerType, nullable = true),
  StructField("DEP_TIME",            DoubleType,  nullable = true),
  StructField("DEP_DELAY",           DoubleType,  nullable = true),
  StructField("TAXI_OUT",            DoubleType,  nullable = true),
  StructField("WHEELS_OFF",          DoubleType,  nullable = true),
  StructField("WHEELS_ON",           DoubleType,  nullable = true),
  StructField("TAXI_IN",             DoubleType,  nullable = true),
  StructField("CRS_ARR_TIME",        IntegerType, nullable = true),
  StructField("ARR_TIME",            DoubleType,  nullable = true),
  StructField("ARR_DELAY",           DoubleType,  nullable = true),
  StructField("CANCELLED",           DoubleType,  nullable = true),
  StructField("CANCELLATION_CODE",   StringType,  nullable = true),
  StructField("DIVERTED",            DoubleType,  nullable = true),
  StructField("CRS_ELAPSED_TIME",    DoubleType,  nullable = true),
  StructField("ACTUAL_ELAPSED_TIME", DoubleType,  nullable = true),
  StructField("AIR_TIME",            DoubleType,  nullable = true),
  StructField("FLIGHTS",             DoubleType,  nullable = true),
  StructField("DISTANCE",            DoubleType,  nullable = true),
  StructField("DISTANCE_GROUP",      IntegerType, nullable = true),
  StructField("CARRIER_DELAY",       DoubleType,  nullable = true),
  StructField("WEATHER_DELAY",       DoubleType,  nullable = true),
  StructField("NAS_DELAY",           DoubleType,  nullable = true),
  StructField("SECURITY_DELAY",      DoubleType,  nullable = true),
  StructField("LATE_AIRCRAFT_DELAY", DoubleType,  nullable = true)
))

println(s"✅ Schema definido con ${schemaVuelos.fields.length} campos")

✅ Schema definido con 29 campos


schemaVuelos: StructType = Seq(
  StructField(
    name = "FL_DATE",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "OP_CARRIER",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "OP_CARRIER_FL_NUM",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "ORIGIN",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "DEST",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "CRS_DEP_TIME",
    dataType = IntegerType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "DEP_TIME",
...

In [ ]:

val (dfRaw, tiempoLecturaCSV) = medirTiempo {
  spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .schema(schemaVuelos)
    .csv(s"$rutaCSVRaw/*.csv")
}


val dfVuelos = dfRaw.withColumn("ANIO", substring(col("FL_DATE"), 1, 4).cast(IntegerType))

val totalFilas = dfVuelos.count()

println(s"=== Dataset cargado ===")
println(s"  Filas totales    : $totalFilas")
println(s"  Columnas         : ${dfVuelos.columns.length}")
println(s"  Tiempo de lectura CSV: ${tiempoLecturaCSV} ms")
println()
dfVuelos.printSchema()
dfVuelos.show(3, truncate = true)

=== Dataset cargado ===
  Filas totales    : 18505725
  Columnas         : 30
  Tiempo de lectura CSV: 130 ms

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = tr

dfRaw: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 27 more fields]
tiempoLecturaCSV: Long = 130L
dfVuelos: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 28 more fields]
totalFilas: Long = 18505725L

In [7]:
// CELDA 6.1 — Escritura CSV consolidado
val rutaCSVOut = s"$rutaSalida/consolidado_csv"

val (_, tEscrituraCSV) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .option("header", "true")
    .csv(rutaCSVOut)
}

println(s"✅ CSV escrito en ${tEscrituraCSV} ms")
println(s"   Ruta: $rutaCSVOut")

✅ CSV escrito en 51891 ms
   Ruta: C:/Curso-Scala/datos/airline/salida/consolidado_csv


rutaCSVOut: String = "C:/Curso-Scala/datos/airline/salida/consolidado_csv"
tEscrituraCSV: Long = 51891L

In [8]:
// CELDA 6.2 — Escritura Parquet sin partición
val rutaParquet = s"$rutaSalida/parquet_sin_partir"

val (_, tEscrituraParquet) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .parquet(rutaParquet)
}

println(s"✅ Parquet escrito en ${tEscrituraParquet} ms")
println(s"   Ruta: $rutaParquet")

✅ Parquet escrito en 55928 ms
   Ruta: C:/Curso-Scala/datos/airline/salida/parquet_sin_partir


rutaParquet: String = "C:/Curso-Scala/datos/airline/salida/parquet_sin_partir"
tEscrituraParquet: Long = 55928L

In [9]:
// CELDA 6.3 — Escritura ORC
val rutaORC = s"$rutaSalida/orc"

val (_, tEscrituraORC) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .orc(rutaORC)
}

println(s"✅ ORC escrito en ${tEscrituraORC} ms")
println(s"   Ruta: $rutaORC")

✅ ORC escrito en 55166 ms
   Ruta: C:/Curso-Scala/datos/airline/salida/orc


rutaORC: String = "C:/Curso-Scala/datos/airline/salida/orc"
tEscrituraORC: Long = 55166L

In [11]:
import org.apache.spark.sql.SparkSession

// 1. Intentamos usar la sesión local sin descargar nada
val spark = SparkSession.builder()
  .appName("ModoOffline")
  .master("local[*]")
  .getOrCreate()

println(s"✅ ¡CONECTADO! Versión: ${spark.version}")

// 2. Leemos los archivos que ya tienes en la carpeta
val rutaVuelos = "C:/Curso-Scala/datos/airline/csv_raw/*.csv"
val dfVuelosRaw = spark.read.option("header", "true").csv(rutaVuelos)

dfVuelosRaw.show(5)


✅ ¡CONECTADO! Versión: 3.5.0
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-----

import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.SparkSession@6ca06d11
rutaVuelos: String = "C:/Curso-Scala/datos/airline/csv_raw/*.csv"
dfVuelosRaw: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 26 more fields]

In [13]:
import org.apache.spark.sql.functions._

// 1. Transformamos columnas a tipos correctos (números y booleanos)
val dfVuelosLimpios = dfVuelosRaw.select(
  col("FL_DATE"),
  col("ORIGIN"),
  col("DEST"),
  col("DEP_DELAY").cast("double"),   // Retraso de salida a número
  col("ARR_DELAY").cast("double"),   // Retraso de llegada a número
  col("CANCELLED").cast("int"),      // Cancelado a entero (0 o 1)
  col("DISTANCE").cast("double")     // Distancia a número
)

// 2. Vemos el nuevo esquema para confirmar el cambio
dfVuelosLimpios.printSchema()

// 3. Hacemos un cálculo rápido: ¿Cuál es el retraso promedio por origen?
dfVuelosLimpios
  .groupBy("ORIGIN")
  .avg("DEP_DELAY")
  .orderBy(desc("avg(DEP_DELAY)"))
  .show(10)


root
 |-- FL_DATE: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: integer (nullable = true)
 |-- DISTANCE: double (nullable = true)

+------+------------------+
|ORIGIN|    avg(DEP_DELAY)|
+------+------------------+
|   ENV|             157.0|
|   YNG|              63.0|
|   TKI|              45.0|
|   ART|36.333333333333336|
|   PPG| 35.97506925207756|
|   OTH| 28.42184154175589|
|   OWB|27.635514018691588|
|   HGR| 26.76086956521739|
|   LWB| 25.70921985815603|
|   MMH|25.641196013289036|
+------+------------------+
only showing top 10 rows



import org.apache.spark.sql.functions._
dfVuelosLimpios: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, ORIGIN: string ... 5 more fields]

In [17]:
// CELDA 7 — Parquet particionado por ANIO
val rutaParquetParticionado = s"$rutaSalida/parquet_particionado"

val (_, tEscrituraParticionado) = medirTiempo {
  dfVuelos.write
    .mode("overwrite")
    .partitionBy("ANIO")
    .parquet(rutaParquetParticionado)
}

println(s"✅ Parquet particionado escrito en ${tEscrituraParticionado} ms")
println(s"   Ruta: $rutaParquetParticionado")
println()
println("Estructura de particiones generada:")
println(s"  $rutaParquetParticionado/")

// Listar las subcarpetas generadas
val carpetasAnio = new File(rutaParquetParticionado)
  .listFiles()
  .filter(_.isDirectory)
  .map(_.getName)
  .sorted

carpetasAnio.foreach(c => println(s"    $c/"))

✅ Parquet particionado escrito en 52367 ms
   Ruta: C:/Curso-Scala/datos/airline/salida/parquet_particionado

Estructura de particiones generada:
  C:/Curso-Scala/datos/airline/salida/parquet_particionado/
    ANIO=2016/
    ANIO=2017/
    ANIO=2018/


rutaParquetParticionado: String = "C:/Curso-Scala/datos/airline/salida/parquet_particionado"
tEscrituraParticionado: Long = 52367L
carpetasAnio: Array[String] = Array("ANIO=2016", "ANIO=2017", "ANIO=2018")

In [18]:
// CELDA 8.1 — Lectura completa CSV
val (dfLeidoCSV, tLecturaCSV) = medirTiempo {
  val df = spark.read
    .option("header", "true")
    .schema(schemaVuelos)
    .csv(rutaCSVOut)
  df.count()
  df
}
println(s"✅ Lectura CSV completa: ${tLecturaCSV} ms  | Filas: ${dfLeidoCSV.count()}")

✅ Lectura CSV completa: 1306 ms  | Filas: 18505725


dfLeidoCSV: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 27 more fields]
tLecturaCSV: Long = 1306L

In [19]:
// CELDA 8.2 — Lectura completa Parquet
val (dfLeidoParquet, tLecturaParquet) = medirTiempo {
  val df = spark.read.parquet(rutaParquet)
  df.count()
  df
}
println(s"✅ Lectura Parquet completa: ${tLecturaParquet} ms  | Filas: ${dfLeidoParquet.count()}")

✅ Lectura Parquet completa: 366 ms  | Filas: 18505725


dfLeidoParquet: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 26 more fields]
tLecturaParquet: Long = 366L

In [20]:
// CELDA 8.3 — Lectura completa ORC
val (dfLeidoORC, tLecturaORC) = medirTiempo {
  val df = spark.read.orc(rutaORC)
  df.count()
  df
}
println(s"✅ Lectura ORC completa: ${tLecturaORC} ms  | Filas: ${dfLeidoORC.count()}")

✅ Lectura ORC completa: 325 ms  | Filas: 18505725


dfLeidoORC: org.apache.spark.sql.package.DataFrame = [FL_DATE: string, OP_CARRIER: string ... 28 more fields]
tLecturaORC: Long = 325L

In [22]:
// CELDA 9.1 — Consulta selectiva sobre CSV
val (resultCSV, tConsultaCSV) = medirTiempo {
  spark.read
    .option("header", "true")
    .schema(schemaVuelos)
    .csv(rutaCSVOut)
    .filter(col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta CSV: ${tConsultaCSV} ms — ${resultCSV.length} aerolíneas")

✅ Consulta CSV: 6257 ms — 18 aerolíneas


resultCSV: Array[org.apache.spark.sql.Row] = Array(
  [WN,3929253,4.49],
  [DL,2778985,-0.31],
  [AA,2689711,4.8],
  [OO,2057083,6.25],
  [UA,1734781,3.17],
  [EV,1005896,6.95],
  [B6,867633,10.19],
  [AS,603579,-0.65],
  [NK,461688,6.53],
  [F9,313710,9.51],
  [YX,305990,3.08],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [HA,240086,0.91],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [VX,155637,8.44],
  [G4,95452,9.98]
)
tConsultaCSV: Long = 6257L

In [ ]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

// 1. Definir función de tiempo
def medirTiempo[T](bloque: => T): (T, Long) = {
  val inicio = System.currentTimeMillis()
  val resultado = bloque
  val fin = System.currentTimeMillis()
  (resultado, fin - inicio)
}

// 2. Leer todos los archivos
val rutaCarpetaCSV = "csv_rw/"
val rutaDestinoAvro = "csv_rw/todos_los_vuelos_limpios.avro"

val (dfVuelosRaw, tLectura) = medirTiempo {
  spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(rutaCarpetaCSV)
}

// 3. LIMPIAR COLUMNAS (Quitar espacios y poner guiones bajos)
val columnasLimpias = dfVuelosRaw.columns.map(c => col(c).as(c.replaceAll(" ", "_")))
val dfLimpio = dfVuelosRaw.select(columnasLimpias: _*)
  .withColumn("ANIO", substring(col("FL_DATE"), 1, 4).cast(IntegerType))

// 4. Guardar en Avro sin errores
val (_, tEscrituraAvro) = medirTiempo {
  dfLimpio.write
    .mode("overwrite")
    .format("avro")
    .save(rutaDestinoAvro)
}

println(s"✅ Procesado con éxito.")
println(s"⏱️ Tiempo lectura: $tLectura ms")
println(s"⏱️ Tiempo escritura Avro: $tEscrituraAvro ms")


In [23]:
// CELDA 9.2 — Consulta selectiva sobre Parquet
val (resultParquet, tConsultaParquet) = medirTiempo {
  spark.read
    .parquet(rutaParquet)
    .filter(col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta Parquet: ${tConsultaParquet} ms — ${resultParquet.length} aerolíneas")

✅ Consulta Parquet: 1244 ms — 18 aerolíneas


resultParquet: Array[org.apache.spark.sql.Row] = Array(
  [WN,3929253,4.49],
  [DL,2778985,-0.31],
  [AA,2689711,4.8],
  [OO,2057083,6.25],
  [UA,1734781,3.17],
  [EV,1005896,6.95],
  [B6,867633,10.19],
  [AS,603579,-0.65],
  [NK,461688,6.53],
  [F9,313710,9.51],
  [YX,305990,3.08],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [HA,240086,0.91],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [VX,155637,8.44],
  [G4,95452,9.98]
)
tConsultaParquet: Long = 1244L

In [25]:
// CELDA 9.5 — Consulta sobre Parquet particionado con filtro por año
val anioConsulta = 2018  // ← cambia el año según los datos que hayas descargado

val (resultParticionado, tConsultaParticionado) = medirTiempo {
  spark.read
    .parquet(rutaParquetParticionado)
    .filter(col("ANIO") === anioConsulta && col("CANCELLED") === 0.0)
    .groupBy("OP_CARRIER")
    .agg(
      count("*").as("total_vuelos"),
      round(avg("ARR_DELAY"), 2).as("retraso_medio_min")
    )
    .orderBy(desc("total_vuelos"))
    .collect()
}
println(s"✅ Consulta Parquet particionado (ANIO=$anioConsulta): ${tConsultaParticionado} ms")
println(s"   Aerolíneas encontradas: ${resultParticionado.length}")

✅ Consulta Parquet particionado (ANIO=2018): 594 ms
   Aerolíneas encontradas: 18


anioConsulta: Int = 2018
resultParticionado: Array[org.apache.spark.sql.Row] = Array(
  [WN,1334277,4.52],
  [DL,945755,-0.29],
  [AA,901873,5.43],
  [OO,763527,7.04],
  [UA,616662,5.76],
  [YX,305990,3.08],
  [B6,298591,11.43],
  [MQ,285346,5.36],
  [OH,266587,8.24],
  [AS,243554,-0.5],
  [9E,239562,4.45],
  [YV,209608,8.85],
  [EV,197220,8.8],
  [NK,174441,5.17],
  [F9,117707,14.21],
  [G4,95452,9.98],
  [HA,83473,0.85],
  [VX,17237,1.73]
)
tConsultaParticionado: Long = 594L

In [ ]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

def medirTiempo[T](bloque: => T): (T, Long) = {
  val inicio = System.currentTimeMillis()
  val resultado = bloque
  val fin = System.currentTimeMillis()
  (resultado, fin - inicio)
}

// 2. LECTURA DE CARPETA
val rutaCarpetaCSV = "csv_rw/"
val rutaDestinoAvro = "csv_rw/todos_vuelos_final.avro"

val (dfRaw, tLectura) = medirTiempo {
  spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(rutaCarpetaCSV)
}


val dfLimpio = dfRaw.toDF(dfRaw.columns.map(_.replaceAll("[^a-zA-Z0-9_]", "_")): _*)
  .withColumn("ANIO", substring(col("FL_DATE"), 1, 4).cast(IntegerType))

// 4. ESCRITURA EN AVRO
val (_, tEscrituraAvro) = medirTiempo {
  dfLimpio.write
    .mode("overwrite")
    .format("avro")
    .save(rutaDestinoAvro)
}


println(s"✅ Total de filas: ${dfLimpio.count()}")
println(s"⏱️ Tiempo lectura: $tLectura ms")
println(s"⏱️ Tiempo escritura Avro: $tEscrituraAvro ms")


1 deprecation (since 2.13.0); re-run enabling -deprecation for details, or try -help


✅ Total de filas: 18505725
⏱️ Tiempo lectura: 11383 ms
⏱️ Tiempo escritura Avro: 26495 ms


import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
defined function medirTiempo
rutaCarpetaCSV: String = "csv_rw/"
rutaDestinoAvro: String = "csv_rw/todos_vuelos_final.avro"
dfRaw: org.apache.spark.sql.package.DataFrame = [FL_DATE: date, OP_CARRIER: string ... 26 more fields]
tLectura: Long = 11383L
dfLimpio: org.apache.spark.sql.package.DataFrame = [FL_DATE: date, OP_CARRIER: string ... 27 more fields]
tEscrituraAvro: Long = 26495L

In [26]:
// CELDA 10 — Comparativa de tamaños en disco
val tamCSVRaw         = tamanoBytes(rutaCSVRaw)
val tamCSVOut         = tamanoBytes(rutaCSVOut)
val tamParquet        = tamanoBytes(rutaParquet)
val tamORC            = tamanoBytes(rutaORC)

val tamParticionado   = tamanoBytes(rutaParquetParticionado)

println("=" * 52)
println("   COMPARATIVA DE TAMAÑO EN DISCO")
println("=" * 52)
println(f"  CSV original (raw)          : ${formatoTamano(tamCSVRaw)}%10s")
println(f"  CSV consolidado (Spark out) : ${formatoTamano(tamCSVOut)}%10s")
println(f"  Parquet (sin partición)     : ${formatoTamano(tamParquet)}%10s")
println(f"  ORC                         : ${formatoTamano(tamORC)}%10s")

println(f"  Parquet particionado        : ${formatoTamano(tamParticionado)}%10s")
println("=" * 52)
println()
println("Ratios de compresión respecto al CSV raw:")
if (tamCSVRaw > 0) {
  println(f"  Parquet  : ${tamCSVRaw.toDouble / tamParquet}%.1fx  menos espacio que CSV")
  println(f"  ORC      : ${tamCSVRaw.toDouble / tamORC}%.1fx  menos espacio que CSV")

}

   COMPARATIVA DE TAMAÑO EN DISCO
  CSV original (raw)          :    2,13 GB
  CSV consolidado (Spark out) :    2,24 GB
  Parquet (sin partición)     :  408,38 MB
  ORC                         :  675,52 MB
  Parquet particionado        :  405,04 MB

Ratios de compresión respecto al CSV raw:
  Parquet  : 5,3x  menos espacio que CSV
  ORC      : 3,2x  menos espacio que CSV


tamCSVRaw: Long = 2289841990L
tamCSVOut: Long = 2409346291L
tamParquet: Long = 428220473L
tamORC: Long = 708329153L
tamParticionado: Long = 424714165L

In [27]:
// CELDA 11 — Tabla comparativa de rendimiento
// Sustituye los valores de ejemplo por los que hayas medido en tus celdas anteriores

case class ResultadoFormato(
  formato:         String,
  tEscrituraMs:    Long,
  tLecturaMs:      Long,
  tConsultaMs:     Long,
  tamanoDisco:     String
)

// ⬇️ Rellena con tus valores reales
val resultados = List(
  ResultadoFormato("CSV",                  tEscrituraCSV,         tLecturaCSV,         tConsultaCSV,         formatoTamano(tamCSVOut)),
  ResultadoFormato("Parquet (sin partir)", tEscrituraParquet,     tLecturaParquet,     tConsultaParquet,     formatoTamano(tamParquet)),
  ResultadoFormato("ORC",                  tEscrituraORC,         tLecturaORC,         tConsultaORC,         formatoTamano(tamORC)),
  
  ResultadoFormato("Parquet particionado", tEscrituraParticionado, tConsultaParticionado, tConsultaParticionado, formatoTamano(tamParticionado))
)

println("=" * 85)
println(f"  ${"FORMATO"}%-25s ${"T.ESCRITURA"}%13s ${"T.LECTURA"}%11s ${"T.CONSULTA"}%12s ${"DISCO"}%10s")
println("=" * 85)
resultados.foreach { r =>
  println(f"  ${r.formato}%-25s ${r.tEscrituraMs}%10d ms  ${r.tLecturaMs}%8d ms  ${r.tConsultaMs}%9d ms  ${r.tamanoDisco}%10s")
}
println("=" * 85)

  FORMATO                     T.ESCRITURA   T.LECTURA   T.CONSULTA      DISCO
  CSV                            51891 ms      1306 ms       6257 ms     2,24 GB
  Parquet (sin partir)           55928 ms       366 ms       1244 ms   408,38 MB
  ORC                            55166 ms       325 ms        923 ms   675,52 MB
  Parquet particionado           52367 ms       594 ms        594 ms   405,04 MB


defined class ResultadoFormato
resultados: List[ResultadoFormato] = List(
  ResultadoFormato(
    formato = "CSV",
    tEscrituraMs = 51891L,
    tLecturaMs = 1306L,
    tConsultaMs = 6257L,
    tamanoDisco = "2,24 GB"
  ),
  ResultadoFormato(
    formato = "Parquet (sin partir)",
    tEscrituraMs = 55928L,
    tLecturaMs = 366L,
    tConsultaMs = 1244L,
    tamanoDisco = "408,38 MB"
  ),
  ResultadoFormato(
    formato = "ORC",
    tEscrituraMs = 55166L,
    tLecturaMs = 325L,
    tConsultaMs = 923L,
    tamanoDisco = "675,52 MB"
  ),
  ResultadoFormato(
    formato = "Parquet particionado",
    tEscrituraMs = 52367L,
    tLecturaMs = 594L,
    tConsultaMs = 594L,
    tamanoDisco = "405,04 MB"
  )
)

In [28]:
// CELDA 12 — Explorar el Parquet particionado con Spark SQL
spark.read
  .parquet(rutaParquetParticionado)
  .createOrReplaceTempView("vuelos_lake")

// Consulta 1: verificar que Spark reconoce la partición ANIO
spark.sql("""
  SELECT ANIO, COUNT(*) AS total_vuelos
  FROM vuelos_lake
  GROUP BY ANIO
  ORDER BY ANIO
""").show()

// Consulta 2: top 5 rutas con más retrasos en un año concreto
spark.sql("""
  SELECT
    ORIGIN,
    DEST,
    COUNT(*)                         AS total_vuelos,
    ROUND(AVG(ARR_DELAY), 1)         AS retraso_medio_min,
    ROUND(SUM(CANCELLED), 0)         AS vuelos_cancelados
  FROM vuelos_lake
  WHERE ANIO = 2018
    AND ARR_DELAY > 0
  GROUP BY ORIGIN, DEST
  ORDER BY retraso_medio_min DESC
  LIMIT 10
""").show(truncate = false)

+----+------------+
|ANIO|total_vuelos|
+----+------------+
|2016|     5617658|
|2017|     5674621|
|2018|     7213446|
+----+------------+

+------+----+------------+-----------------+-----------------+
|ORIGIN|DEST|total_vuelos|retraso_medio_min|vuelos_cancelados|
+------+----+------------+-----------------+-----------------+
|RDM   |MFR |1           |1347.0           |0.0              |
|MDT   |HPN |1           |798.0            |0.0              |
|GRK   |ATL |6           |243.8            |0.0              |
|ISP   |MSP |7           |224.0            |0.0              |
|ICT   |DAY |1           |210.0            |0.0              |
|JFK   |JAC |2           |199.0            |0.0              |
|DSM   |PIA |1           |168.0            |0.0              |
|TVC   |EWR |52          |165.5            |0.0              |
|EGE   |IAH |22          |165.3            |0.0              |
|ERI   |ITH |1           |160.0            |0.0              |
+------+----+------------+--------------

Sobre los formatos:

¿Qué formato tardó más en escribirse? ¿A qué crees que se debe?El formato que más tardo es Parquet 
¿Qué formato ocupó menos espacio en disco? ¿Por qué los formatos columnares comprimen mejor que CSV?orc  agrupan los datos en el mismo tipo juntos .
¿Qué formato fue más rápido para la consulta que solo usaba 3 columnas? Parket¿Puedes explicar por qué usando el concepto de almacenamiento columnar?porque solo lee esas 3 columnas y ignoran al resto .
Avro es binario y tiene schema embebido, pero fue más lento en la consulta selectiva que Parquet y ORC. ¿Sabes por qué?POrque tiene que entrar por tpdp el archivo e ignora lo demás.
Sobre el particionado:

¿Qué diferencia de tiempo observaste entre la consulta sobre Parquet sin particionar y la consulta con filtro ANIO=2018 sobre el Parquet particionado? le parqeut sin patrocinar lee todos los archivps de todos los años tarda más en cambio el partocinado va directamente a los archivos del 2018.
¿Qué ocurriría con el tiempo de consulta si en lugar de particionar por ANIO hubieras particionado por OP_CARRIER (aerolínea)?Colapsaaria el sistema de archivos y hace que el sisteja taaarde mucho más con los archivos.
En producción, ¿por qué no es recomendable particionar por una columna con demasiados valores únicos?Sobre los formatos:

¿Qué formato tardó más en escribirse? ¿A qué crees que se debe?El formato que más tardo es Parquet 
¿Qué formato ocupó menos espacio en disco? ¿Por qué los formatos columnares comprimen mejor que CSV?orc  agrupan los datos en el mismo tipo juntos .
¿Qué formato fue más rápido para la consulta que solo usaba 3 columnas? Parket¿Puedes explicar por qué usando el concepto de almacenamiento columnar?porque solo lee esas 3 columnas y ignoran al resto .
Avro es binario y tiene schema embebido, pero fue más lento en la consulta selectiva que Parquet y ORC. ¿Sabes por qué?POrque tiene que entrar por tpdp el archivo e ignora lo demás.
Sobre el particionado:

¿Qué diferencia de tiempo observaste entre la consulta sobre Parquet sin particionar y la consulta con filtro ANIO=2018 sobre el Parquet particionado? le parqeut sin patrocinar lee todos los archivps de todos los años tarda más en cambio el partocinado va directamente a los archivos del 2018.
¿Qué ocurriría con el tiempo de consulta si en lugar de particionar por ANIO hubieras particionado por OP_CARRIER (aerolínea)?Colapsaaria el sistema de archivos y hace que el sisteja taaarde mucho más con los archivos.
En producción, ¿por qué no es recomendable particionar por una columna con demasiados valores únicos?Sobre los formatos:

¿Qué formato tardó más en escribirse? ¿A qué crees que se debe?El formato que más tardo es Parquet 
¿Qué formato ocupó menos espacio en disco? ¿Por qué los formatos columnares comprimen mejor que CSV?orc  agrupan los datos en el mismo tipo juntos .
¿Qué formato fue más rápido para la consulta que solo usaba 3 columnas? Parket¿Puedes explicar por qué usando el concepto de almacenamiento columnar?porque solo lee esas 3 columnas y ignoran al resto .
Avro es binario y tiene schema embebido, pero fue más lento en la consulta selectiva que Parquet y ORC. ¿Sabes por qué?POrque tiene que entrar por tpdp el archivo e ignora lo demás.
Sobre el particionado:

¿Qué diferencia de tiempo observaste entre la consulta sobre Parquet sin particionar y la consulta con filtro ANIO=2018 sobre el Parquet particionado? le parqeut sin patrocinar lee todos los archivps de todos los años tarda más en cambio el partocinado va directamente a los archivos del 2018.
¿Qué ocurriría con el tiempo de consulta si en lugar de particionar por ANIO hubieras particionado por OP_CARRIER (aerolínea)?QUe el sistema iria mucho mas lento ya que tendría que entrar en todas las carpetas de las aerolíneas.
En producción, ¿por qué no es recomendable particionar por una columna con demasiados valores únicos?Porque colpasaria el sistema  y tardaria ms tiempo abrinedo los achivos .

Sobre el pipeline:

¿Por qué es mejor definir el schema manualmente en lugar de usar inferSchema = true con ficheros de 600 MB?El esquema manual ahorra una lectura completa ya que es necesario cuando los datos crecen 
Si el Ministerio de Transporte añade los datos de 2019, ¿qué modo de escritura usarías para incorporarlos al Parquet particionado sin borrar los años anteriores.Usaria el append ya que nos permite añadir una nuvea carpeta sin tocar ni borrar los datos de los años 
Sobre el pipeline:

¿Por qué es mejor definir el schema manualmente en lugar de usar inferSchema = true con ficheros de 600 MB?
Si el Ministerio de Transporte añade los datos de 2019, ¿qué modo de escritura usarías para incorporarlos al Parquet particionado sin borrar los años anteriores?